In [1]:
import igraph
#print(igraph.__version__)

In [ ]:
import os
import pickle
import pandas as pd
import time
import networkx as nx
import igraph as ig
import leidenalg as la
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("metrics_processing.log"),
        logging.StreamHandler()
    ]
)

In [11]:
graph_dir = "/home/imw-mmi/Projects/NLP/semantic_networks/igraph_matrices"
output_igraph_dir = "/home/imw-mmi/Projects/NLP/semantic_networks/igraph_outputs"
os.makedirs(output_igraph_dir, exist_ok=True)

In [ ]:
#metrics_csv= os.path.join(output_igraph_dir, "igraph_metrics_summary_full_v2.csv")
metrics_csv= os.path.join(output_igraph_dir, "igraph_metrics_summary_full_v3.csv")

if os.path.exists(metrics_csv):
    df_existing = pd.read_csv(metrics_csv)
    processed_langs = set(df_existing["language"].tolist())
    metrics_summary = df_existing.to_dict(orient="records")  # keep previous entries
    logging.info(f"Loaded existing summary with {len(processed_langs)} languages")
    #print(f"existing summary with {len(processed_langs)} languages")
else:
    df_existing = pd.DataFrame()
    processed_langs = set()
    metrics_summary = []
    logging.info("No previous summary file found. Starting fresh.")
    #print("No previous metrics summary found")


# Limit to 5 languages
#max_langs = 6
#processed = 0
#metrics_summary = []

i = 0

for file in os.listdir(graph_dir):
    # if not file.endswith("_graph.pkl"):
    #     continue

    # lang_code = file.split("_")[0]
    # file_path = os.path.join(graph_dir, file) 

    print(file)
    #sq.top10k.embeddings.simw.cosine_sim.mknn100.graphml

    lang_code = file.split(".")[0]
    method = file.split(".")[-2]

    # start for threshold
    #if method == "mknn100":
    #    continue

    # if i < 5:
    #     i+=1
    # else:
    #     i+=1
    #     break


    # Skip if already converted

    # graphml_path = os.path.join(output_igraph_dir, f"{lang_code}_igraph.graphml")
    #if os.path.exists(graphml_path):
    #    print(f"Skipping {lang_code}: GraphML already exists")
    #    continue

    # if lang_code in processed_langs and method in processed_langs:
    #     logging.info(f"Skipping {lang_code}: already in CSV summary.")
    #     continue

    #print(f"\n processing: {lang_code}..")
    #file_path = os.path.join(graph_dir, file)
    start_time = time.time()

    logging.info(f" Processing language: {lang_code}")

    file_path =  os.path.join(graph_dir, file)

    try:
        # Load from NetworkX pickle
        # with open(file_path, "rb") as f:
        #     G_nx = pickle.load(f)

        # if G_nx.number_of_nodes() == 0 or G_nx.number_of_edges() == 0:
        #     print("Empty graph. Skipping.")
        #     continue

        # Convert to igraph
        # G_ig = ig.Graph.Adjacency((nx.to_numpy_array(G_nx) > 0).tolist(), mode="undirected")

        
        G_ig = ig.Graph.Read_GraphML(file_path)#nx.read_graphml(file_path)

        logging.info(f" Converted NetworkX to igraph for {lang_code} | "
                     f"Nodes: {G_ig.vcount()}, Edges: {G_ig.ecount()}")



        #def print_igraph_conversion_summary(G_nx, G_ig, lang_code="N/A"):
        #    print(f"\nConverted NetworkX to igraph for {lang_code}")
        #    print(f"NetworkX: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")
        #    print(f"iGraph: {G_ig.vcount()} nodes, {G_ig.ecount()} edges")
        #    print(f"iGraph summary: {G_ig.summary()}")
        #print_igraph_conversion_summary(G_nx, G_ig, lang_code)

        # 1. Basic
        #num_nodes = G_ig.vcount()
        #num_edges = G_ig.ecount()
        #density = G_ig.density()
        #degrees = G_ig.degree()
        #avg_degree = float(np.mean(degrees))
        #median_degree = float(np.median(degrees))
        #degree_std = float(np.std(degrees))
        #max_degree = int(np.max(degrees))

        # 2. Distance metrics (only if connected)
        #if G_ig.is_connected():
        #    diameter = G_ig.diameter()
        #    avg_path_len = G_ig.average_path_length()
        #else:
        #    diameter = None
        #    avg_path_len = None

        
        # 2. Distance metrics on the giant component
        components = G_ig.components() # find connected components
        num_components = len(components) # number of connected components
        giant = components.giant() # largest connected component as a subgraph
        largest_component_nodes = giant.vcount() # number of nodes in the largest subgraph
        largest_component_ratio = largest_component_nodes / num_nodes if num_nodes > 0 else 0 # ratio of nodes in the largest subgraph to total nodes

        #if giant.vcount() > 1: # need at least 2 nodes to compute diameter and avg path length
        #    diameter_gc = giant.diameter() # longest shortest path in the giant component
        #    avg_shortest_path_gc = giant.average_path_length() # average shortest path length in the giant component
        #else:
        #    diameter_gc = None # if the giant component has 0 or 1 node, diameter and avg path length are not defined
        #    avg_shortest_path_gc = None 
        

        # 3. Clustering
        #avg_local_clustering = G_ig.transitivity_avglocal_undirected()
        #global_transitivity = G_ig.transitivity_undirected()

        #for the giant component
        avg_local_clustering = giant.transitivity_avglocal_undirected()
        global_transitivity = giant.transitivity_undirected()


        # 4. Community detection (Leiden algorithm) 
        # Note: Using Leiden algorithm instead of Louvain for better performance
        #try:
        #    partition = la.find_partition(G_ig, la.ModularityVertexPartition)
        #    modularity = partition.modularity
        #    num_communities = len(partition)
        #    #louvain = G_ig.community_multilevel()
        #    #modularity = louvain.modularity
        #    #num_communities = len(louvain)
        #except Exception as e:
        #    logging.error(f" Leiden failed for {lang_code}: {e}")
        #    modularity = None
        #    num_communities = None

        #For the giant component
        try:
            partition = la.find_partition(giant, la.ModularityVertexPartition)
            modularity = partition.modularity
            num_communities = len(partition)
        except Exception as e:
            logging.error(f" Leiden failed for {lang_code}: {e}")
            modularity = None
            num_communities = None

        # 5. Assortativity (for the whole graph)
        #try:
        #    assortativity = G_ig.assortativity_degree()
        #except Exception as e:
        #    logging.warning(f"Assortativity failed for {lang_code}: {e}")
        #    assortativity = None

        #For the giant component

        try:
            assortativity = giant.assortativity_degree()
        except Exception as e:
            logging.warning(f"Assortativity failed for {lang_code}: {e}")
            assortativity = None

        # 6. Max betweenness
        #try:
        #    max_betweenness = max(G_ig.betweenness())
        #except Exception as e:
        #    logging.warning(f"Betweenness failed for {lang_code}: {e}")
        #    max_betweenness = None


        # Save metrics
        metrics_summary.append({
            "language": lang_code,
            "method": method,
            #"nodes": num_nodes,
            #"edges": num_edges,
            #"density": density,
            #"avg_degree": avg_degree,
            #"median_degree": median_degree,
            #"degree_std": degree_std,
            #"max_degree": max_degree,
            "num_components": num_components,
            "largest_component_nodes": largest_component_nodes,
            "largest_component_ratio": largest_component_ratio,
            #"diameter": diameter_gc,
            #"avg_shortest_path": avg_shortest_path_gc,
            "avg_clustering_coefficient": avg_local_clustering,
            "transitivity": global_transitivity,
            "modularity": modularity,
            "num_communities": num_communities,
            "degree_assortativity": assortativity,
            #"max_betweenness": max_betweenness
        })

        # Save CSV incrementally
        #pd.DataFrame(metrics_summary).to_csv(metrics_csv, index=False)

        # Save as GraphML

        #print(f"{lang_code} done in {time.time() - start_time:.2f}s — saved to {graphml_path}")

        # if G_ig.vcount() > 0 and G_ig.ecount() > 0:
        #     #graphml_path = os.path.join(output_igraph_dir, f"{lang_code}_igraph.graphml")
        #     G_ig.write_graphml(graphml_path)
        #     logging.info(f" Saved GraphML for {lang_code} to {graphml_path}")
        #     #print(f"{lang_code} done in {time.time() - start_time:.2f}s — saved to {graphml_path}")
        # else:
        #     logging.warning(f"{lang_code} graph is empty.GraphML not saved.")

        # Save CSV 
        pd.DataFrame(metrics_summary).to_csv(metrics_csv, index=False)
        
        elapsed_time = time.time() - start_time
        logging.info(f"Finished processing {lang_code} in {elapsed_time:.2f}s")


        #processed += 1
        #if processed >= max_langs:
        #    print("\n Reached max language count.")
        #    break

    except Exception as e:
        logging.error(f" Error in {lang_code}: {e}")

df_final = pd.DataFrame(metrics_summary)
df_final.to_csv(metrics_csv, index=False)
logging.info(f"\n Metrics summary saved to: {metrics_csv}")

2026-04-18 17:40:24,893 [INFO] No previous summary file found. Starting fresh.
2026-04-18 17:40:24,896 [INFO]  Processing language: it


sq.top10k.embeddings.simw.cosine_sim.mknn100.graphml
it.top10k.embeddings.simw.cosine_sim.thr_p95.graphml


2026-04-18 17:40:27,025 [INFO]  Converted NetworkX to igraph for it | Nodes: 10000, Edges: 2499751
2026-04-18 17:42:53,907 [INFO] Finished processing it in 149.01s
2026-04-18 17:42:53,908 [INFO]  Processing language: is


is.top10k.embeddings.simw.cosine_sim.thr_p95.graphml


2026-04-18 17:42:55,914 [INFO]  Converted NetworkX to igraph for is | Nodes: 10000, Edges: 2499751
2026-04-18 17:46:33,351 [INFO] Finished processing is in 219.44s
2026-04-18 17:46:33,351 [INFO]  Processing language: ro


...

bg.top10k.embeddings.simw.cosine_sim.thr_p95.graphml


2026-04-18 17:48:47,008 [INFO]  Converted NetworkX to igraph for bg | Nodes: 10000, Edges: 2499751
2026-04-18 17:53:07,828 [INFO] Finished processing bg in 262.79s
2026-04-18 17:53:07,829 [INFO]  Processing language: ca


ca.top10k.embeddings.simw.cosine_sim.thr_p95.graphml


2026-04-18 17:53:09,838 [INFO]  Converted NetworkX to igraph for ca | Nodes: 10000, Edges: 2499751
2026-04-18 17:56:56,453 [INFO] Finished processing ca in 228.62s
2026-04-18 17:56:56,453 [INFO]  Processing language: lt


lt.top10k.embeddings.simw.cosine_sim.thr_p95.graphml
